# Device Placement and Vectorization

Trainer note: we teach one rule repeatedly - **move once, compute many times**.

**Outcome:** learners can spot slow loop patterns and rewrite them.

## Step 1 - Setup in tiny chunks

Pause and ask: where should tensors live for repeated operations?

In [ ]:
import time
import torch

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## Step 2 - Loop-heavy pattern (baseline)

Expected: this style is easy to write, but can be slower.

In [ ]:
vals = torch.randn(20000, device=device)
out = torch.zeros_like(vals)

In [ ]:
t0 = time.perf_counter()
for i in range(vals.numel()):
    out[i] = vals[i] * 2.0
loop_time = time.perf_counter() - t0

## Step 3 - Vectorized rewrite

Common mistake: moving tensors between CPU and GPU inside loops.

In [ ]:
t1 = time.perf_counter()
vec = vals * 2.0
vec_time = time.perf_counter() - t1

In [ ]:
print("loop:", round(loop_time, 4))
print("vec :", round(vec_time, 4))

## Checkpoint — Which change removed the biggest bottleneck?

**The questions we asked:**
- Which line removed the biggest bottleneck?
- Where would this pattern appear in real ML code?

**Model answer (strong understanding)**

The single biggest win almost always comes from moving the tensors to the GPU *once*, before the loop, and then leaving them there for the entire repeated computation. Every time you do `tensor.to(device)` or create a fresh tensor inside the loop and copy it, you pay PCIe transfer cost (roughly 10-20 GB/s on a good connection) plus Python overhead for each iteration. That cost dwarfs the actual matrix math for small-to-medium work.

The vectorized `matmul` rewrite is nice for readability and sometimes for kernel fusion, but the *placement* change ("create on device" or "move once before the loop") is the one that typically delivers the order-of-magnitude improvement in this kind of micro-benchmark.

In real ML code this pattern shows up everywhere: embedding tables and model weights live on GPU for the entire training run; optimizer states live on GPU; you do not copy the batch to device inside the training loop on every step; you allocate the large activation tensors on the device once and reuse buffers. The people who accidentally do host-to-device copies inside their training loop are the ones wondering why their 8-GPU box is no faster than a laptop.

**If you thought the matmul rewrite was the hero change, you are in excellent company** — it feels like the "smart" thing. The deeper and more common win is almost always "stop moving the damn data back and forth."

**Common misconception**

"I will just put everything on the GPU at the beginning of the script and it will be fast."

This is half right. Putting the model and the big persistent tensors on the GPU is correct. The trap is assuming that *all* data movement is now free. Large batch copies still cost real time and memory bandwidth. The disciplined engineer still thinks about *when* and *how much* data crosses the bus, even after the "move once" rule is applied.

**If your timings still looked bad after the rewrite**

Check whether the "bad" version was accidentally running on CPU (easy to do when experimenting). Also check whether you had a warm-up effect on the first run. Re-run the cells a couple of times. The important learning is the *shape* of the improvement, not the absolute numbers on this particular Colab instance.


## Lesson Recap — What You Actually Learned

- The dominant cost in many "GPU is slow" stories is not the arithmetic — it is the repeated movement of data between host and device.
- Creating tensors directly on the target device (or moving them once before a hot loop) is a first-order performance technique, not a micro-optimization.
- Vectorization and kernel choice matter, but placement decisions usually matter more for end-to-end time in the regimes you will actually care about.
- You now have a visceral sense for why "just port your numpy code to CuPy" sometimes feels disappointing until you also fix the data movement pattern.

**Human note:** This is the section where a lot of people quietly realize they have been writing GPU code for months that was accidentally CPU-bound because of hidden copies. If that just landed for you, good. You are now ahead of most practitioners.


## Role Lens — Why This Matters in Real Work

**DevOps / MLOps** — When a team says "we moved our training to GPUs and it got slower," the root cause is very often repeated host-to-device copies inside the training or data-loading loop. Your job becomes teaching them the "move once" rule and then proving it with a 5-minute profiling session using the exact mental model you just built.

**Data Science** — Feature engineering that runs in a pandas loop and then gets copied to the GPU on every epoch is a classic silent killer. The difference between a 4-hour training run and a 35-minute one is frequently just "stop copying the feature matrix 200 times."

**Data Engineering** — GPU ETL pipelines (RAPIDS, Polars on GPU, etc.) live and die by the same rule. If your extraction step is still producing CPU objects that you then ship to the GPU inside the transformation loop, you are paying the tax on every record. The engineers who internalize "create the cuDF object once, transform in place" ship pipelines that actually finish before the next data arrives.

---

**You have internalized the single most important rule in practical GPU programming.**

Next we apply this rule at scale to real data-science workloads (training loops, mixed precision, batch tuning). The foundation you just built is what makes the later sections click instead of feeling like magic.
